In [ ]:
import json
import re
from datetime import date, timedelta
from pathlib import Path

import pandas as pd
import spacy

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

DATA_DIR = Path("../data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)

nlp = spacy.load("en_core_web_sm")

## 1. Load sentences

In [ ]:
sent_df = pd.read_csv(DATA_DIR / "segmented_sentences.csv")
sent_df["publication_date"] = pd.to_datetime(sent_df["publication_date"], errors="coerce")
sent_df = sent_df.sort_values(["article_id", "sentence_id"]).reset_index(drop=True)

print("Total sentences:", len(sent_df))
print("Articles       :", sent_df["article_id"].nunique())
sent_df.head()

## 2. Lexicons
Small, easy-to-edit word lists that control what counts as an event.

In [ ]:
# "X said that <event>"  ->  the <event> is kept, X becomes `attributed_to`
NEUTRAL_SPEECH = {
    "say", "tell", "report", "state", "add", "note", "mention", "claim",
    "explain", "confirm", "announce", "indicate", "acknowledge", "reveal", "disclose",
}

# Communicative acts that are news events themselves ("warned", "forecast"):
# the whole clause is kept as ONE event.
ACT_VERBS = {
    "warn", "forecast", "predict", "advise", "urge", "order", "decide",
    "declare", "instruct", "caution", "appeal",
}

# "-ing" verbs that only link a cause to its result ("causing X to flood").
CAUSAL_LEMMAS = {"cause", "lead", "force", "prompt", "trigger", "result"}

# verbs after which a bare infinitive is part of the same event ("helped residents leave")
CAUSATIVE_VERBS = {"help", "let", "make", "have", "see", "watch", "hear", "allow", "enable"}

SUBJ_DEPS = {"nsubj", "nsubjpass", "csubj", "csubjpass", "nsubj:pass"}
OBJ_DEPS = {"dobj", "obj", "dative", "attr", "oprd"}
REL_PRON = {"which", "who", "whom", "that", "whose"}

LOC_PREPS = {"in", "at", "near", "across", "toward", "towards", "from", "inside",
             "outside", "along", "into", "within", "around"}
TEMPORAL_NOUNS = {"hour", "minute", "second", "day", "week", "month", "year", "morning",
                  "afternoon", "evening", "night", "noon", "midnight", "overnight",
                  "today", "tomorrow", "yesterday", "time", "moment", "period"}
GENERIC_PLACES = {"city", "area", "river", "location", "site", "place", "street", "road",
                  "home", "town", "region", "country", "premises", "spot"}
PARTITIVES = {"part", "parts", "area", "areas", "side", "section", "sections",
              "centre", "center", "edge", "heart", "outskirts", "portion", "portions"}

MONTHS = {m: i for i, m in enumerate(
    ["january", "february", "march", "april", "may", "june", "july",
     "august", "september", "october", "november", "december"], 1)}
MONTHS.update({"jan": 1, "feb": 2, "mar": 3, "apr": 4, "jun": 6, "jul": 7,
               "aug": 8, "sep": 9, "sept": 9, "oct": 10, "nov": 11, "dec": 12})
WEEKDAYS = {d: i for i, d in enumerate(
    ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"])}
RANGE_PREPS = {"through", "until", "till", "thru"}      # "through Saturday" = END of a period, not the event date
NUMBER_WORDS = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7,
                "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12, "twenty": 20,
                "thirty": 30, "forty": 40, "fifty": 50, "hundred": 100}
DURATION_UNITS = {"second", "seconds", "minute", "minutes", "hour", "hours", "day", "days",
                  "week", "weeks", "month", "months", "year", "years"}
# approximate minute-of-day, used ONLY to order events inside one day
PART_OF_DAY = {"overnight": 120, "dawn": 330, "morning": 540, "noon": 720, "afternoon": 900,
               "evening": 1140, "dusk": 1100, "night": 1260, "midnight": 0}
PART_MODIFIER = {("early", "morning"): 360, ("late", "morning"): 660, ("early", "afternoon"): 780,
                 ("late", "afternoon"): 1020, ("early", "evening"): 1080, ("late", "evening"): 1260,
                 ("early", "night"): 1200, ("late", "night"): 1380}
FUTURE_AUX = {"will", "would", "shall"}
FUTURE_LEMMAS = {"forecast", "expect", "predict", "plan", "schedule"}

## 3. Helper functions: phrases, subjects, objects

In [ ]:
def tidy(text):
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    return text.strip(" ,;:")


def strip_edges(tokens):
    """Remove leading/trailing punctuation, conjunctions and clause markers."""
    toks = list(tokens)
    while toks and (toks[0].is_punct or toks[0].pos_ in ("CCONJ", "SCONJ")):
        toks.pop(0)
    while toks and (toks[-1].is_punct or toks[-1].pos_ in ("CCONJ", "SCONJ")):
        toks.pop()
    return toks


def join_tokens(tokens, override=None):
    """Join tokens keeping the original spacing; a gap between non-adjacent tokens becomes one space.
    `override` = {token_index: replacement_text or None(drop)}."""
    override = override or {}
    parts, prev = [], None
    for t in tokens:
        if t.i in override and override[t.i] is None:
            continue
        if prev is not None:
            parts.append(prev.whitespace_ if t.i == prev.i + 1 else " ")
        parts.append(override.get(t.i, t.text))
        prev = t
    return tidy("".join(parts))


IRREGULAR_PAST = {"flee": "fled", "leave": "left", "rise": "rose", "begin": "began", "hold": "held",
                  "make": "made", "take": "took", "bring": "brought", "cut": "cut", "shut": "shut",
                  "set": "set", "put": "put", "lose": "lost", "become": "became", "go": "went",
                  "get": "got", "give": "gave", "run": "ran", "fall": "fell", "break": "broke",
                  "drive": "drove", "send": "sent", "build": "built", "keep": "kept", "lead": "led",
                  "stand": "stood", "shut": "shut", "hit": "hit", "burst": "burst", "spread": "spread"}


def past_tense(verb):
    """Small rule-based past tense, used only for 'X to flood' -> 'X flooded'."""
    v = verb.lower()
    if v in IRREGULAR_PAST:
        return IRREGULAR_PAST[v]
    if v.endswith("e"):
        return v + "d"
    if v.endswith("y") and len(v) > 1 and v[-2] not in "aeiou":
        return v[:-1] + "ied"
    return v + "ed"


def is_causal_participle(tok):
    """'causing X to flood', 'leading authorities to close'  (but NOT 'the hours leading up to Friday')."""
    if tok.tag_ != "VBG" or tok.lemma_.lower() not in CAUSAL_LEMMAS:
        return False
    nxt = tok.doc[tok.i + 1] if tok.i + 1 < len(tok.doc) else None
    return not (nxt is not None and nxt.lower_ == "up")


def np_tokens(tok, drop_relcl=True, drop_conj=False):
    """Tokens of the noun phrase headed by `tok` (without relative clauses / appositions;
    optionally without coordinated phrases: 'School and Hall' -> 'School')."""
    ids = {t.i for t in tok.subtree}
    if drop_relcl:
        for t in tok.subtree:
            if t.dep_ in ("relcl", "appos") and t.i != tok.i:
                ids -= {x.i for x in t.subtree}
    if drop_conj:
        for t in tok.children:
            if t.dep_ in ("conj", "cc"):
                ids -= {x.i for x in t.subtree}
    toks = [tok.doc[i] for i in sorted(ids)]
    toks = [t for t in toks if not is_causal_participle(t)]
    return strip_edges(toks)


def np_text(tok):
    return join_tokens(np_tokens(tok))


def causal_control_np(verb):
    """'..., leading authorities to close ...'  ->  the NP right after a causal participle."""
    doc = verb.doc
    for t in reversed(doc[:verb.i]):
        if is_causal_participle(t):
            for nxt in doc[t.i + 1: verb.i]:
                if nxt.pos_ in ("NOUN", "PROPN"):
                    # climb to the head of the noun phrase
                    head = nxt
                    while head.head.i > head.i and head.dep_ in ("compound", "amod", "nmod") and head.head.pos_ in ("NOUN", "PROPN"):
                        head = head.head
                    return t, head
            return t, None
        if t.dep_ == "ROOT":
            break
    return None, None


def has_aux_to(verb):
    return any(c.dep_ == "aux" and c.lower_ == "to" for c in verb.children)


def has_own_subject(verb):
    return any(c.dep_ in SUBJ_DEPS for c in verb.children)

## 4. Which verbs are events?

In [ ]:
def classify_predicate(tok):
    """
    Returns (role, type) where role is
      'event'    -> becomes its own event
      'merged'   -> part of its head's event (xcomp, purpose clause, ...)
      'speech'   -> neutral reporting verb whose complement is the real event
      None       -> ignore
    """
    if tok.pos_ not in ("VERB", "AUX"):
        return None, None
    if tok.pos_ == "AUX" and tok.dep_ != "ROOT":
        return None, None
    if is_causal_participle(tok):
        return None, None

    lemma = tok.lemma_.lower()
    comps = [c for c in tok.children if c.dep_ in ("ccomp", "xcomp") and c.pos_ in ("VERB", "AUX")]

    # neutral reporting verb with a clause -> the clause is the event
    if lemma in NEUTRAL_SPEECH and any(c.dep_ == "ccomp" for c in comps):
        return "speech", "speech"

    head = tok.head
    dep = tok.dep_

    if dep == "ROOT":
        return "event", "main"
    if dep == "conj":
        return "event", "coordinated"
    if dep == "parataxis":
        return "event", "parallel"
    if dep == "relcl":
        return ("event", "relative_clause") if tok.tag_ in ("VBD", "VBN", "VBZ", "VBP") else (None, None)
    if dep == "advcl":
        if has_aux_to(tok) and not has_own_subject(tok) and not any(c.dep_ == "mark" for c in tok.children):
            return "merged", "purpose"                      # "... to assess the situation"
        return "event", "subordinate"
    if dep in ("ccomp", "xcomp"):
        if is_causal_participle(head):
            return "event", "result_of_cause"               # "causing streets to flood"
        if head.lemma_.lower() in NEUTRAL_SPEECH and dep == "ccomp":
            return "event", "reported_statement"            # "said that 46 residents had been moved"
        if head.lemma_.lower() in ACT_VERBS:
            return "merged", "complement"                   # "warned that rain was expected"
        if dep == "xcomp" or head.lemma_.lower() in CAUSATIVE_VERBS:
            return "merged", "complement"                   # "continued to deal", "helped ... leave"
        return "event", "complement"
    return None, None

## 5. Temporal expressions (clause level)

The scanner reads only the tokens that belong to one event and returns a small dictionary:
month/day, weekday, relative words, clock time, part of day, early/late modifier, and
whether the expression is the **end of a period** (`through Saturday`, `until Saturday evening`)
or only a **duration** (`twelve hours`), neither of which is the event date.

In [ ]:
def to_int(text):
    text = text.lower()
    if text.isdigit():
        return int(text)
    return NUMBER_WORDS.get(text)


def introduced_by_range(tok):
    """True when the token sits under 'through / until / till'."""
    cur, hops = tok, 0
    while hops < 3:
        if cur.head.lower_ in RANGE_PREPS:
            return True
        if cur.head.i == cur.i:
            break
        cur, hops = cur.head, hops + 1
    return False


def modifier_before(doc, start, allowed_gap=3):
    for j in range(start - 1, max(start - 1 - allowed_gap, -1), -1):
        w = doc[j].lower_
        if w in ("early", "late"):
            return w
        if w not in ("on", "at", "by", "the", "in", "of", "until", "through", "around", "approximately"):
            break
    return None


def scan_temporal(doc, idxs):
    out = {"month_day": None, "year": None, "weekday": None, "relative": None, "relative_days": None,
           "clock": None, "clock_text": None, "part": None, "modifier": None,
           "range_end": {"month_day": None, "weekday": None, "part": None, "modifier": None},
           "duration": [], "expressions": [], "used": set()}
    toks = [doc[i] for i in sorted(idxs)]
    n = len(doc)

    for k, t in enumerate(toks):
        low = t.lower_
        nxt = doc[t.i + 1] if t.i + 1 < n else None
        nxt2 = doc[t.i + 2] if t.i + 2 < n else None

        # ---- clock time: "5:30 a.m.", "10 a.m."
        if re.fullmatch(r"\d{1,2}(:\d{2})?", low) and nxt is not None and nxt.lower_ in ("a.m.", "p.m.", "am", "pm", "a.m", "p.m"):
            hh, mm = (int(x) for x in (low.split(":") + ["0"])[:2])
            pm = nxt.lower_.startswith("p")
            hh = (hh % 12) + (12 if pm else 0)
            if out["clock"] is None:
                out["clock"] = (hh, mm)
                out["clock_text"] = f"{t.text} {nxt.text}"
                out["expressions"].append(out["clock_text"])
            out["used"] |= {t.i, nxt.i}
            continue

        # ---- month + day:  "August 13"  /  "13 August"
        if low in MONTHS:
            day_tok = None
            if nxt is not None and re.fullmatch(r"\d{1,2}(st|nd|rd|th)?", nxt.lower_):
                day_tok = nxt
            elif t.i > 0 and re.fullmatch(r"\d{1,2}(st|nd|rd|th)?", doc[t.i - 1].lower_):
                day_tok = doc[t.i - 1]
            if day_tok is not None:
                md = (MONTHS[low], int(re.sub(r"\D", "", day_tok.text)))
                if introduced_by_range(t):
                    out["range_end"]["month_day"] = md
                elif out["month_day"] is None:
                    out["month_day"] = md
                    out["modifier"] = out["modifier"] or modifier_before(doc, min(t.i, day_tok.i))
                    out["expressions"].append(f"{t.text} {day_tok.text}")
                out["used"] |= {t.i, day_tok.i}
                yr = doc[max(t.i, day_tok.i) + 1] if max(t.i, day_tok.i) + 1 < n else None
                if yr is not None and re.fullmatch(r"(19|20)\d{2}", yr.text):
                    out["year"] = int(yr.text)
                    out["used"].add(yr.i)
                continue

        # ---- weekday
        if low in WEEKDAYS:
            if introduced_by_range(t):
                out["range_end"]["weekday"] = WEEKDAYS[low]
                out["range_end"]["modifier"] = modifier_before(doc, t.i)
                if nxt is not None and nxt.lower_ in PART_OF_DAY:
                    out["range_end"]["part"] = nxt.lower_
                    out["used"].add(nxt.i)
            elif out["weekday"] is None:
                out["weekday"] = WEEKDAYS[low]
                out["modifier"] = out["modifier"] or modifier_before(doc, t.i)
                out["expressions"].append(t.text)
            out["used"].add(t.i)
            continue

        # ---- relative words
        if low in ("yesterday", "today", "tonight", "tomorrow") and t.dep_ != "poss":
            out["relative"] = low
            out["expressions"].append(t.text)
            out["used"].add(t.i)
            continue
        if low == "day" and k > 0 and toks[k - 1].lower_ == "previous" and t.dep_ != "poss":
            out["relative"] = "yesterday"
            out["expressions"].append("the previous day")
            out["used"] |= {t.i, toks[k - 1].i}
            continue
        if low == "day" and t.dep_ == "poss":                 # "the previous day's downpour" -> modifies a noun, not the event
            out["used"].add(t.i)
            if k > 0:
                out["used"].add(toks[k - 1].i)
            continue

        # ---- durations / "N hours ago"
        if to_int(low) is not None and nxt is not None and nxt.lower_ in DURATION_UNITS:
            if nxt2 is not None and nxt2.lower_ == "ago":
                out["relative_days"] = -(to_int(low) if nxt.lower_.startswith("day") else 0)
                out["expressions"].append(f"{t.text} {nxt.text} ago")
                out["used"] |= {t.i, nxt.i, nxt2.i}
            else:
                out["duration"].append(f"{t.text} {nxt.text}")
                out["used"] |= {t.i, nxt.i}
            continue

        # ---- part of day ("morning", "noon", "overnight")
        if low in PART_OF_DAY and t.i not in out["used"]:
            if introduced_by_range(t):
                out["range_end"]["part"] = low
            else:
                if out["part"] is None:
                    out["part"] = low
                    out["modifier"] = out["modifier"] or modifier_before(doc, t.i, 2)
                    out["expressions"].append(t.text)
                if low == "noon" and out["clock"] is None:
                    out["clock"], out["clock_text"] = (12, 0), "noon"
            out["used"].add(t.i)

    return out

## 6. Resolving a temporal expression to a calendar date

* the article's **publication date** is the anchor
* `Friday` -> the most recent Friday on/before the publication date (future tense -> the next one)
* `August 13` -> that date in the publication year
* `through / until X` is stored as `period_end`, **never** as the event date
* durations such as `twelve hours` are ignored for dating

In [ ]:
def resolve_weekday(pub, weekday, future=False):
    if future:
        return pub + timedelta(days=(weekday - pub.weekday()) % 7)
    return pub - timedelta(days=(pub.weekday() - weekday) % 7)


def resolve_month_day(pub, month, day, year=None):
    try:
        d = date(year or pub.year, month, day)
    except ValueError:
        return None
    if year is None and d - pub > timedelta(days=45):
        d = date(pub.year - 1, month, day)
    return d


def sort_minutes(sc):
    if sc["clock"]:
        return sc["clock"][0] * 60 + sc["clock"][1]
    part, mod = sc["part"], sc["modifier"]
    if part:
        if (mod, part) in PART_MODIFIER:
            return PART_MODIFIER[(mod, part)]
        return PART_OF_DAY[part]
    if mod == "late":
        return 1320
    if mod == "early":
        return 360
    return None


def time_label(sc):
    if sc["clock_text"]:
        return sc["clock_text"]
    if sc["part"]:
        return f"{sc['modifier']} {sc['part']}" if sc["modifier"] else sc["part"]
    if sc["modifier"]:
        return sc["modifier"]
    return None


def resolve_temporal(sc, pub, future):
    """-> (date | None, source | None, period_end | None)"""
    d, src = None, None
    if sc["month_day"]:
        d = resolve_month_day(pub, *sc["month_day"], year=sc["year"])
        src = "explicit_date"
    elif sc["relative"]:
        offset = {"yesterday": -1, "today": 0, "tonight": 0, "tomorrow": 1}[sc["relative"]]
        d, src = pub + timedelta(days=offset), "relative_date"
    elif sc["relative_days"] is not None:
        d, src = pub + timedelta(days=sc["relative_days"]), "relative_date"
    elif sc["weekday"] is not None:
        d, src = resolve_weekday(pub, sc["weekday"], future), "weekday"

    end = None
    re_ = sc["range_end"]
    if re_["month_day"]:
        end = resolve_month_day(pub, *re_["month_day"])
    elif re_["weekday"] is not None:
        end = resolve_weekday(pub, re_["weekday"], future=True)
    return d, src, end

## 7. Locations (clause level)

In [ ]:
def clean_location(tok):
    """Noun phrase of a place, without determiners; 'parts of Riverside' -> 'Riverside'."""
    # partitive: "parts of Riverside", "the eastern part of the city"
    if tok.lemma_.lower() in PARTITIVES:
        for c in tok.children:
            if c.dep_ == "prep" and c.lower_ == "of":
                for g in c.children:
                    if g.dep_ == "pobj" and g.pos_ == "PROPN":
                        return np_text(g)
    toks = np_tokens(tok, drop_conj=True)
    while toks and (toks[0].pos_ == "DET" or toks[0].dep_ == "poss"):
        toks.pop(0)
    return join_tokens(strip_edges(toks))


def extract_locations(doc, idxs, temporal_used):
    found = []
    for i in sorted(idxs):
        t = doc[i]
        if t.dep_ != "prep" or t.lower_ not in LOC_PREPS:
            continue
        for c in t.children:
            if c.dep_ != "pobj":
                continue
            for node in [c] + list(c.conjuncts):
                if node.i not in idxs:
                    continue
                if {x.i for x in node.subtree} & temporal_used:
                    continue
                if node.lemma_.lower() in TEMPORAL_NOUNS or node.like_num or node.pos_ == "NUM":
                    continue
                phrase = clean_location(node)
                if not phrase:
                    continue
                words = phrase.split()
                is_proper = any(w[0].isupper() for w in words)
                if not is_proper and len(words) == 1 and node.lemma_.lower() in GENERIC_PLACES:
                    continue
                if not is_proper and words[-1].lower() in GENERIC_PLACES and len(words) <= 2 and words[0].lower() in {"both", "two", "the", "these", "those"}:
                    continue
                if phrase.lower() in {"both sites", "two locations", "the two locations"}:
                    continue
                found.append(phrase)
    # named entities that were not found through a preposition
    for ent in doc.ents:
        if ent.label_ in ("GPE", "LOC", "FAC") and all(x.i in idxs for x in ent):
            txt = join_tokens([x for x in ent if x.pos_ != "DET"])
            if txt and txt.lower() not in {f.lower() for f in found} and not any(txt.lower() in f.lower() for f in found):
                found.append(txt)
    uniq, seen = [], set()
    for f in found:
        if f.lower() not in seen:
            seen.add(f.lower())
            uniq.append(f)
    return uniq

## 8. Quantities (numbers with their noun)

In [ ]:
def extract_quantities(doc, idxs, temporal_used):
    """'118 millimetres', '46 residents', '60 people' - numbers that are not dates/times."""
    out = []
    for i in sorted(idxs):
        t = doc[i]
        if i in temporal_used or t.ent_type_ == "ORDINAL":
            continue
        if (t.like_num or t.tag_ == "CD") and t.dep_ == "nummod" and t.head.pos_ in ("NOUN", "PROPN"):
            noun = t.head
            if noun.lemma_.lower() in GENERIC_PLACES | {"location", "site", "place"}:
                continue
            num = t.text if t.text.isdigit() else str(to_int(t.lower_) or t.text)
            out.append(f"{num} {noun.text}")
    return list(dict.fromkeys(out))

## 9. Build events for one sentence

In [ ]:
def merged_children(verb, merged_ids):
    for c in verb.children:
        if c.i in merged_ids:
            yield c
            yield from merged_children(c, merged_ids)


def future_tense(verb, merged_ids):
    vs = [verb] + list(merged_children(verb, merged_ids))
    for v in vs:
        if any(c.dep_ == "aux" and c.lower_ in FUTURE_AUX for c in v.children):
            return True
        if v.lemma_.lower() in FUTURE_LEMMAS:
            return True
    return False


def extract_sentence_events(doc):
    skipped = {t.i for t in doc if is_causal_participle(t)}
    roles = {}
    for t in doc:
        role, etype = classify_predicate(t)
        if role:
            roles[t.i] = (role, etype)

    event_ids = {i for i, (r, _) in roles.items() if r == "event"}
    merged_ids = {i for i, (r, _) in roles.items() if r == "merged"}
    speech_ids = {i for i, (r, _) in roles.items() if r == "speech"}

    # sentence consisting only of a speech verb with no usable complement -> treat the verb as the event
    for i in list(speech_ids):
        if not any(c.i in event_ids for c in doc[i].children):
            speech_ids.discard(i)
            event_ids.add(i)
            roles[i] = ("event", "main")

    def ancestors_events(tok):
        return [a for a in tok.ancestors if a.i in event_ids]

    # -------- subjects (and which tokens the subject "consumes")
    info, consumed = {}, set()
    for i in sorted(event_ids):
        v = doc[i]
        subs, source, sub_ids = [], "missing", set()
        direct = [c for c in v.children if c.dep_ in SUBJ_DEPS]
        if direct:
            for s in direct:
                if s.lower_ in REL_PRON and v.dep_ == "relcl":
                    subs.append(np_text(v.head)); sub_ids |= {x.i for x in np_tokens(v.head)}; source = "relative_antecedent"
                else:
                    subs.append(np_text(s)); sub_ids |= {x.i for x in np_tokens(s)}; source = "direct"
        elif v.dep_ == "relcl":
            subs, source = [np_text(v.head)], "relative_antecedent"
        else:
            ctrl_tok, ctrl_np = causal_control_np(v) if (has_aux_to(v) or v.dep_ in ("conj", "advcl", "ccomp")) else (None, None)
            if ctrl_np is not None and has_aux_to(v):
                subs, source = [np_text(ctrl_np)], "controlled_by_cause"
                consumed |= {x.i for x in np_tokens(ctrl_np)}
            elif v.dep_ in ("conj", "advcl", "parataxis") and not has_aux_to(v):
                parents = ancestors_events(v)
                head = v.head if v.head.i in event_ids else (parents[0] if parents else None)
                if head is not None:
                    inherited = [c for c in head.children if c.dep_ in SUBJ_DEPS]
                    if inherited:
                        subs, source = [np_text(inherited[0])], "inherited"
            elif v.dep_ in ("ccomp", "xcomp"):
                head = v.head
                objs = [c for c in head.children if c.dep_ in ("dobj", "obj", "dative")]
                subj = [c for c in head.children if c.dep_ in SUBJ_DEPS]
                if objs:
                    subs, source = [np_text(objs[0])], "object_control"
                elif subj:
                    subs, source = [np_text(subj[0])], "inherited"
        info[i] = {"subjects": [s for s in subs if s], "source": source, "sub_ids": sub_ids}

    # -------- token scope of every event
    scopes = {}
    for i in sorted(event_ids):
        v = doc[i]
        ids = {t.i for t in v.subtree}
        for j in event_ids:
            if j != i and j in ids and any(a.i == i for a in doc[j].ancestors):
                sub = sorted(t.i for t in doc[j].subtree)
                lo, hi = sub[0], sub[-1]
                ids -= set(sub)
                if lo > 0 and doc[lo - 1].text == ",":            # ", which had been ..., was restored"
                    ids.discard(lo - 1)
                    if hi + 1 < len(doc) and doc[hi + 1].text == ",":
                        ids.discard(hi + 1)
        ids -= skipped
        scopes[i] = ids
    for i in scopes:                       # NP consumed by a child event is removed from the parent
        owner_children = [j for j in event_ids if j != i and info[j]["source"] == "controlled_by_cause"]
        for j in owner_children:
            if not (consumed & {t.i for t in doc[j].subtree}):
                scopes[i] -= consumed

    events = []
    for i in sorted(event_ids):
        v = doc[i]
        etype = roles[i][1]
        ids = scopes[i]
        toks = [doc[k] for k in sorted(ids)]
        # drop a leading causal tail such as "leading" when it survived the parse
        toks = [t for t in toks if not is_causal_participle(t)]
        if v.dep_ == "relcl":                                  # "which had been suspended" -> "had been suspended"
            toks = [t for t in toks if not (t.lower_ in REL_PRON and t.dep_ in SUBJ_DEPS)]
        toks = strip_edges(toks)
        override = {}
        causal_before = [t for t in doc[:v.i] if t.i in skipped]      # "..., leading authorities to close ..."
        if (etype == "result_of_cause" or info[i]["source"] == "controlled_by_cause" or causal_before) and has_aux_to(v) and v.tag_ == "VB":
            # "several streets ... to flood" -> "several streets ... flooded"
            for c in v.children:
                if c.dep_ == "aux" and c.lower_ == "to":
                    override[c.i] = None
            override[v.i] = past_tense(v.text)
        text = join_tokens(toks, override)

        subj_info = info[i]
        if subj_info["subjects"] and not (subj_info["sub_ids"] and subj_info["sub_ids"] <= ids) and subj_info["source"] in ("controlled_by_cause", "inherited", "object_control", "relative_antecedent"):
            text = f"{subj_info['subjects'][0]} {text}"
        text = tidy(text)
        if len(text.split()) < 3:
            continue
        text = text[0].upper() + text[1:]

        # attribution ("Officials said that ...")
        attributed = ""
        if etype == "reported_statement":
            speaker = [c for c in v.head.children if c.dep_ in SUBJ_DEPS]
            attributed = np_text(speaker[0]) if speaker else ""

        # objects (also from merged complements such as "began distributing X")
        group = [v] + list(merged_children(v, merged_ids))
        objects = []
        for g in group:
            objects += [np_text(c) for c in g.children if c.dep_ in OBJ_DEPS]
        objects = list(dict.fromkeys(o for o in objects if o))

        agents = []
        for c in v.children:
            if c.dep_ in ("agent", "obl:agent"):
                agents += [np_text(g) for g in c.children if g.dep_ == "pobj"]

        subj_heads = [c for c in v.children if c.dep_ in SUBJ_DEPS]
        passive = any(c.dep_ in ("nsubjpass", "nsubj:pass") for c in v.children) or any(c.dep_ == "auxpass" for c in v.children)
        negation = (any(c.dep_ == "neg" for c in v.children)
                    or any(d.lower_ in ("no", "neither", "never", "none") for s in subj_heads for d in s.children if d.dep_ in ("det", "neg", "advmod")))

        causal = [t.text.lower() for t in doc if t.i in skipped and any(a.i == t.i for a in v.ancestors)]
        if info[i]["source"] == "controlled_by_cause" or (causal_before and has_aux_to(v)):
            causal = causal or [t.text.lower() for t in causal_before]

        sc = scan_temporal(doc, ids)
        locations = extract_locations(doc, ids, sc["used"])
        quantities = extract_quantities(doc, ids, sc["used"])

        parent = next((a.i for a in v.ancestors if a.i in event_ids), None)

        events.append({
            "trigger_index": i,
            "parent_trigger_index": parent,
            "event_trigger": v.text,
            "action": v.lemma_.lower(),
            "event_type": etype,
            "event_text": text,
            "subject": subj_info["subjects"],
            "subject_source": subj_info["source"],
            "object": objects,
            "agent": list(dict.fromkeys(agents)),
            "passive": bool(passive),
            "negation": bool(negation),
            "attributed_to": attributed,
            "causal_indicators": causal,
            "quantities": quantities,
            "location_entities": locations,
            "temporal_expressions": sc["expressions"],
            "durations": sc["duration"],
            "_scan": sc,
            "_future": future_tense(v, merged_ids),
        })
    return events

## 10. Quick test on the sentences that were problematic before

In [ ]:
tests = [
    "Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.",
    "Heavy rainfall continued in Riverton on Friday, August 14, leading authorities to close all municipal schools for the day.",
    "Officials said that 46 residents had been moved to the shelters by 10 a.m.",
    "The weather office warned that further rainfall was expected through Saturday.",
    "Power supply, which had been suspended in parts of Riverside as a safety measure, was restored to most streets by 11 a.m.",
]
for s in tests:
    print("\nSENTENCE:", s)
    for e in extract_sentence_events(nlp(s)):
        print(f"  [{e['event_type']:<18}] {e['event_text']}")
        print(f"      subject={e['subject']} object={e['object']} attributed_to={e['attributed_to']!r}")
        print(f"      locations={e['location_entities']} time_expr={e['temporal_expressions']} durations={e['durations']}")

## 11. Run on every sentence and resolve event dates

In [ ]:
rows = []
for _, r in sent_df.iterrows():
    doc = nlp(r["sentence"])
    for ev in extract_sentence_events(doc):
        rows.append({"article_id": r["article_id"], "sentence_id": r["sentence_id"],
                     "publication_date": r["publication_date"], "original_sentence": r["sentence"], **ev})

events_df = pd.DataFrame(rows).reset_index(drop=True)
events_df.insert(0, "event_id", range(1, len(events_df) + 1))
print("Total events:", len(events_df))

In [ ]:
def assign_dates(df):
    """Adds event_date, event_time, sort_minutes, date_source, date_confidence, period_end."""
    out = []
    by_key = {}
    for idx, row in df.iterrows():
        pub = row["publication_date"].date()
        sc = row["_scan"]
        d, src, end = resolve_temporal(sc, pub, row["_future"])
        minutes = sort_minutes(sc)
        label = time_label(sc)
        conf = "explicit" if d is not None else None
        if d is None:                                     # inherit from the parent event in the same sentence
            parent = by_key.get((row["article_id"], row["sentence_id"], row["parent_trigger_index"]))
            if parent is not None and parent["confidence"] == "explicit":
                d, src, conf = parent["date"], "inherited_from_parent_event", "inferred"
        if d is None:
            d = pub
            if minutes is not None:
                src, conf = "publication_date_with_time", "time_only"
            else:
                src, conf = "publication_date", "inferred"
        by_key[(row["article_id"], row["sentence_id"], row["trigger_index"])] = {"date": d, "confidence": conf}
        out.append({"event_date": d, "event_time": (f"{minutes // 60:02d}:{minutes % 60:02d}" if minutes is not None else None),
                    "event_time_label": label, "sort_minutes": minutes, "date_source": src,
                    "date_confidence": conf, "period_end": end})
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(out)], axis=1)


events_df = assign_dates(events_df)
events_df["event_date"] = pd.to_datetime(events_df["event_date"])
events_df["period_end"] = pd.to_datetime(events_df["period_end"])
events_df["pub_weekday"] = events_df["publication_date"].dt.day_name()

events_df[["event_id", "article_id", "event_text", "event_date", "event_time_label", "date_source", "date_confidence"]]

## 12. Sanity checks

In [ ]:
print("Events per article:")
print(events_df.groupby("article_id").size(), "\n")

for col in ["subject", "object", "location_entities"]:
    empty = events_df[col].apply(lambda v: len(v) == 0).sum()
    print(f"{col:<18} empty in {empty} / {len(events_df)} events")

print("\nDate sources:")
print(events_df["date_source"].value_counts())
print("\nEvents whose date is more than 3 days away from the publication date (should be none):")
far = events_df[(events_df["event_date"] - events_df["publication_date"]).abs() > pd.Timedelta(days=3)]
display(far[["event_id", "article_id", "event_text", "event_date"]])

## 13. Save

In [ ]:
LIST_COLS = ["subject", "object", "agent", "causal_indicators", "quantities",
             "location_entities", "temporal_expressions", "durations"]
save_df = events_df.drop(columns=["_scan", "_future"]).copy()
for c in LIST_COLS:
    save_df[c] = save_df[c].apply(json.dumps)
save_df["event_date"] = save_df["event_date"].dt.strftime("%Y-%m-%d")
save_df["period_end"] = save_df["period_end"].dt.strftime("%Y-%m-%d")
save_df["publication_date"] = save_df["publication_date"].dt.strftime("%Y-%m-%d")

save_df.to_csv(DATA_DIR / "extracted_events_final.csv", index=False)
print(f"Saved {len(save_df)} events to {DATA_DIR / 'extracted_events_final.csv'}")